# Homework 3 Notebook 
Author: Eugenia Tate 

Course: AIPI 530 by Dr. Natalia Summerville 

## Problem 1 — Campus courier tour (TSP)

A courier starts at office **O**, visits locations **A, B, and C** exactly once, and returns to O while minimizing the total travel distance.

#### Loading packages

In [140]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

#### Creating Data
Locations: O, A, B, and C. Office O is the starting and ending point.
The distances are the same in both directions. The model uses directed arcs to represent travel between locations.

In [141]:
O, A, B, C = "O", "A", "B", "C"
DEPOT = O

distance_data = pd.DataFrame(columns=["From", "To", "Distance"])

distance_data["From"] = [O, O, O, O, A, A, A, A, B, B, B, B, C, C, C, C]

distance_data["To"] = [O, A, B, C, O, A, B, C, O, A, B, C, O, A, B, C]

distance_data["Distance"] = [0, 7, 12, 9, 7, 0, 6, 8, 12, 6, 0, 5, 9, 8, 5, 0]

distance_data

,From,To,Distance
0,O,O,0
1,O,A,7
2,O,B,12
3,O,C,9
4,A,O,7
5,A,A,0
6,A,B,6
7,A,C,8
8,B,O,12
9,B,A,6


#### Define Model

In [142]:
# create empty model
m = AbstractModel()

**Sets and parameters**

- $V = \{O, A, B, C\}$: set of campus locations
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs (no self-loops)
- $d_{ij}$: distance from location $i$ to location $j$ in miles
- $n = |V| = 4$: total number of locations

In [143]:
# define sets
m.V = Set()
m.A = Set(
    within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j]
)

# define parameters
m.d = Param(m.V, m.V)
m.n = Param(initialize=lambda m: len(m.V))

**Decision variables**

In [144]:
m.x = Var(m.A, domain=Binary)

m.u = Var(
    m.V,
    domain=NonNegativeIntegers,
    bounds=lambda m, i: (0, 0) if i == DEPOT else (1, m.n - 1),
)

#### Constraints

**Leave each location exactly once**

In [145]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1

m.LeaveOnce = Constraint(m.V, rule=Leave_Once_Rule)

**Enter each location exactly once**

In [146]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1

m.EnterOnce = Constraint(m.V, rule=Enter_Once_Rule)

**Subtour elimination (MTZ)**

The MTZ constraints prevent separate loops by requiring the visit order to increase when the courier travels directly between two non-depot locations.

In [147]:
def MTZ_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip

    return m.u[i] - m.u[j] + (m.n - 1) * m.x[i, j] <= m.n - 2

m.MTZ = Constraint(m.A, rule=MTZ_Rule)

**Objective: Minimize total distance**

In [148]:
def Min_Distance_Rule(m):
    return sum(m.d[i, j] * m.x[i, j] for (i, j) in m.A)

m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

#### Create Instance

In [149]:
instanceData = {
    None: {
        "V": {None: list(distance_data["From"].unique())},
        "d": distance_data.set_index(["From", "To"])["Distance"].to_dict(),
    }
}

instance = m.create_instance(instanceData)

#### Solve

In [150]:
# solve instance
solver = SolverFactory("glpk")
solution = solver.solve(instance)

print("Solver Status:")
print(solution["Solver"])

print("Optimum Objective Function Value:")
print(value(instance.Min_Distance))

Solver Status:

- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 5
      Number of created subproblems: 5
  Error rc: 0
  Time: 0.014508962631225586

Optimum Objective Function Value:
27.0


#### Print solution

In [151]:
# create output data: follow the arcs from the depot to get the tour in order
next_city = {i: j for (i, j) in instance.A if value(instance.x[i, j]) > 0.5}

tour = [DEPOT]
while next_city[tour[-1]] != DEPOT:
    tour.append(next_city[tour[-1]])
tour.append(DEPOT)

output_route = pd.DataFrame(
    [
        {"Stop": k + 1, "From": i, "To": j, "Distance": value(instance.d[i, j])}
        for k, (i, j) in enumerate(zip(tour[:-1], tour[1:]))
    ]
)

print("===== OPTIMAL TOUR =====")
print(" → ".join(tour))
print(f"Total Distance: {value(instance.Min_Distance):.0f}")

print("\nSelected decision variables:")
for i, j in instance.A:
    if value(instance.x[i, j]) > 0.5:
        print(f"x[{i},{j}] = {value(instance.x[i,j]):.0f}")

# print ordering variables, excluding the depot
print("\nOrdering variables:")
for i in instance.V:
    if i != DEPOT:
        print(f"u[{i}] = {value(instance.u[i]):.0f}")

output_route

===== OPTIMAL TOUR =====
O → A → B → C → O
Total Distance: 27

Selected decision variables:
x[O,A] = 1
x[A,B] = 1
x[B,C] = 1
x[C,O] = 1

Ordering variables:
u[A] = 1
u[B] = 2
u[C] = 3


,Stop,From,To,Distance
0,1,O,A,7
1,2,A,B,6
2,3,B,C,5
3,4,C,O,9


#### Explanation of Results

The optimal route is O → A → B → C → O, with a total travel distance of 27 miles. The reverse route is also optimal since the distances are same.

The route visits each campus location exactly once and returns to the dispatch office. The arrival and departure constraints are satisfied, and the MTZ constraints prevent disconnected subtours.

The solver reports an optimal solution - it is not just a local optimum - that is 27 miles is the minimum possible distance.

## Problem 2 - Campus Collection Routes (CVRP)

Two identical trucks start at office O and collect boxes from locations A, B, and C. Each location must be visited exactly once, and both trucks must return to O.

Each truck has a capacity of 10 boxes. The goal is to minimize the total distance traveled by both trucks.

In [152]:
# define packages
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

#### Creating Data

The model uses the same campus locations and distances as Problem 1. Pickup quantities and truck capacity are added for this problem.

In [153]:
# distance data
O, A, B, C = "O", "A", "B", "C"
DEPOT = O

distance_data = pd.DataFrame(columns=["From", "To", "Distance"])
distance_data["From"] = [O, O, O, O, A, A, A, A, B, B, B, B, C, C, C, C]
distance_data["To"] = [O, A, B, C, O, A, B, C, O, A, B, C, O, A, B, C]
distance_data["Distance"] = [0, 7, 12, 9, 7, 0, 6, 8, 12, 6, 0, 5, 9, 8, 5, 0]
distance_data

,From,To,Distance
0,O,O,0
1,O,A,7
2,O,B,12
3,O,C,9
4,A,O,7
5,A,A,0
6,A,B,6
7,A,C,8
8,B,O,12
9,B,A,6


In [154]:
# pickup and fleet data
demand_data = pd.DataFrame(columns=["Customer", "Pickup"])

demand_data["Customer"] = [A, B, C]
demand_data["Pickup"] = [8, 6, 4]

TRUCKS = 2
CAPACITY = 10

demand_data

,Customer,Pickup
0,A,8
1,B,6
2,C,4


#### Defining Model

In [155]:
# create empty model
m = AbstractModel()

**Sets and Parameters**
- $V$: set of locations (depot and customers)
- $C = V \setminus \{O\}$: set of customers
- $A = \{(i,j) \in V \times V : i \ne j\}$: set of arcs (no self-loops)
- $c_{ij}$: distance from location $i$ to location $j$
- $d_i$: quantity picked up at customer $i$
- $K$: number of trucks
- $Q$: truck capacity

In [156]:
# define sets
m.V = Set()

m.C = Set(within=m.V, initialize=lambda m: [i for i in m.V if i != DEPOT])

m.A = Set(
    within=m.V * m.V, initialize=lambda m: [(i, j) for i in m.V for j in m.V if i != j]
)

# define parameters
m.c = Param(m.V, m.V)
m.d = Param(m.C)
m.K = Param()
m.Q = Param()

**Decision Variables**

In [157]:
# define decision variables
m.x = Var(m.A, domain=Binary)
m.u = Var(m.C, domain=NonNegativeReals, bounds=lambda m, i: (m.d[i], m.Q))

#### Constraints

Each customer must have one arrival

In [158]:
def Enter_Once_Rule(m, j):
    return sum(m.x[i, j] for i in m.V if i != j) == 1

m.EnterOnce = Constraint(m.C, rule=Enter_Once_Rule)

Each customer must have one departure

In [159]:
def Leave_Once_Rule(m, i):
    return sum(m.x[i, j] for j in m.V if j != i) == 1

m.LeaveOnce = Constraint(m.C, rule=Leave_Once_Rule)

Two trucks leave the depot

In [160]:
def Depot_Departures_Rule(m):
    return sum(m.x[DEPOT, j] for j in m.C) == m.K

m.DepotDepartures = Constraint(rule=Depot_Departures_Rule)

Two trucks return to the depot

In [161]:
def Depot_Returns_Rule(m):
    return sum(m.x[i, DEPOT] for i in m.C) == m.K

m.DepotReturns = Constraint(rule=Depot_Returns_Rule)

Load propagation: enforce truck capacity and prevent subtours

In [162]:
def Load_Rule(m, i, j):
    if i == DEPOT or j == DEPOT:
        return Constraint.Skip

    return m.u[i] - m.u[j] + m.Q * m.x[i, j] <= m.Q - m.d[j]

m.Load = Constraint(m.A, rule=Load_Rule)

**Objective:** minimize total dist

In [163]:
def Min_Distance_Rule(m):
    return sum(m.c[i, j] * m.x[i, j] for (i, j) in m.A)


m.Min_Distance = Objective(rule=Min_Distance_Rule, sense=minimize)

#### Create the Instance


In [164]:
# build and read instance data
instanceData = {
    None: {
        "V": {None: list(distance_data["From"].unique())},
        "c": distance_data.set_index(["From", "To"])["Distance"].to_dict(),
        "d": demand_data.set_index("Customer")["Pickup"].to_dict(),
        "K": {None: TRUCKS},
        "Q": {None: CAPACITY},
    }
}

instance = m.create_instance(instanceData)

#### Solve

In [165]:
# solve instance
solver = SolverFactory("glpk")
solution = solver.solve(instance)

print("Solver Status:")
print(solution["Solver"])

print("Optimum Objective Function Value:")
print(value(instance.Min_Distance))

Solver Status:

- Status: ok
  Termination condition: optimal
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: 3
      Number of created subproblems: 3
  Error rc: 0
  Time: 0.008868932723999023

Optimum Objective Function Value:
40.0


#### Print solution

In [166]:
# follow the selected arcs to build both routes
next_city = {
    i: j for (i, j) in instance.A if i != DEPOT and value(instance.x[i, j]) > 0.5
}

first_stops = [j for j in instance.C if value(instance.x[DEPOT, j]) > 0.5]

rows = []

print("===== OPTIMAL ROUTES =====")

for truck, first in enumerate(first_stops, start=1):
    route = [DEPOT, first]

    while route[-1] != DEPOT:
        route.append(next_city[route[-1]])

    load = 0

    for k, (i, j) in enumerate(zip(route[:-1], route[1:])):
        load += value(instance.d[j]) if j != DEPOT else 0

        rows.append(
            {
                "Truck": truck,
                "Stop": k + 1,
                "From": i,
                "To": j,
                "Distance": value(instance.c[i, j]),
                "Load": load,
            }
        )

    route_distance = sum(value(instance.c[i, j]) for i, j in zip(route[:-1], route[1:]))

    print(
        f"Truck {truck}: "
        + " -> ".join(route)
        + f"   Load: {load}   Distance: {route_distance}"
    )

print(f"\nTotal Distance: {value(instance.Min_Distance):.0f}")

output_routes = pd.DataFrame(rows)
output_routes

===== OPTIMAL ROUTES =====
Truck 1: O -> A -> O   Load: 8   Distance: 14
Truck 2: O -> C -> B -> O   Load: 10   Distance: 26

Total Distance: 40


,Truck,Stop,From,To,Distance,Load
0,1,1,O,A,7,8
1,1,2,A,O,7,8
2,2,1,O,C,9,4
3,2,2,C,B,5,10
4,2,3,B,O,12,10


In [167]:
# print selected routing variables
print("Selected routing variables:")
for i, j in instance.A:
    if value(instance.x[i, j]) > 0.5:
        print(f"x[{i},{j}] = {value(instance.x[i,j]):.0f}")

# print load variables
print("\nLoad variables:")
for i in instance.C:
    print(f"u[{i}] = {value(instance.u[i]):.2f}")

Selected routing variables:
x[O,A] = 1
x[O,C] = 1
x[A,O] = 1
x[B,O] = 1
x[C,B] = 1

Load variables:
u[A] = 8.00
u[B] = 10.00
u[C] = 4.00


### Explanation of Results

The optimal solution uses two trucks with a minimum combined travel distance of **40 miles**.

One truck travels O → A → O, collecting 8 boxes over 14 miles. The other travels O → B → C → O, collecting 10 boxes over 26 miles.

All three locations are visited exactly once, and both trucks return to the depot. The second truck reaches its full capacity of 10 boxes, while the first truck has 2 boxes of unused capacity.

The solver reports an optimal solution, confirming that 40 miles is the minimum total distance.

## Problem 3 - One-Week Clearance Plan

The goal is to find the optimal discounts for three fabric items to maximize revenue after disposal costs.

#### Loading Packages

In [168]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory, SolverStatus, TerminationCondition

#### Create model and define sets and decision variables

In [169]:
# define model
m = ConcreteModel()

# define sets
m.ITEMS = Set(initialize=["A", "B", "C"])

# define decision variable
m.MarkPct = Var(m.ITEMS, domain=NonNegativeReals, bounds=(0, 1))

# define implicit variables
m.MarkPrice = Var(m.ITEMS, domain=NonNegativeReals)
m.Demand = Var(m.ITEMS, domain=NonNegativeReals)
m.Revenue = Var(m.ITEMS, domain=NonNegativeReals)
m.FinalInv = Var(m.ITEMS, domain=NonNegativeReals)

# define params
m.basePrice = Param(m.ITEMS, initialize={"A": 24, "B": 32, "C": 40})
m.initInv = Param(m.ITEMS, initialize={"A": 12, "B": 24, "C": 36})
m.maxPct = 0.80
m.cap = 12
m.dispCost = 1.5

# define equations for implicit variables

# define markdown price
def Define_MarkPrice_Rule(m, i):
    return (1 - m.MarkPct[i]) * m.basePrice[i] == m.MarkPrice[i]

m.Define_MarkPrice = Constraint(m.ITEMS, rule=Define_MarkPrice_Rule)

# define demand
def Define_Demand_Rule(m, i):
    return m.MarkPct[i] * m.initInv[i] == m.Demand[i]

m.Define_Demand = Constraint(m.ITEMS, rule=Define_Demand_Rule)

# define revenue
def Define_Revenue_Rule(m, i):
    return m.MarkPrice[i] * m.Demand[i] == m.Revenue[i]

m.Define_Revenue = Constraint(m.ITEMS, rule=Define_Revenue_Rule)

# define final inventory
def Define_FinalInv_Rule(m, i):
    return m.initInv[i] - m.Demand[i] == m.FinalInv[i]

m.Define_FinalInv = Constraint(m.ITEMS, rule=Define_FinalInv_Rule)

#### Constraints

In [170]:
# define max discount
def Max_Discount_Rule(m, i):
    return m.MarkPct[i] <= m.maxPct

m.Max_Discount = Constraint(m.ITEMS, rule=Max_Discount_Rule)

# limit leftover inventory for each item
def Leftover_Inv_Rule(m, i):
    return m.FinalInv[i] <= m.cap

m.Leftover_Inv = Constraint(m.ITEMS, rule=Leftover_Inv_Rule)

# define objective function
def Max_Revenue_Rule(m):
    return sum((m.Revenue[i] - m.dispCost * m.FinalInv[i]) for i in m.ITEMS)

m.Max_Revenue = Objective(rule=Max_Revenue_Rule, sense=maximize)

#### Solve

In [171]:
solver = SolverFactory("ipopt")
solution = solver.solve(m)

#### Print Solution 

In [172]:
print(solution["Solver"])



- Status: ok
  Message: Ipopt 3.14.20\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.34776806831359863



##### Print optimum value of objective function

In [173]:
print(m.Max_Revenue.expr())

539.7031264174957


Print optimum value of decision and implicit variables 

In [174]:
data = {
    "ITEM": [],
    "Opt_Discount": [],
    "Opt_Price": [],
    "Opt_Sales": [],
    "Opt_Revenue": [],
    "Final_Inv": [],
}

for i in m.ITEMS:
    data["ITEM"].append(i)
    data["Opt_Discount"].append(m.MarkPct[i].value)
    data["Opt_Price"].append(m.MarkPrice[i].value)
    data["Opt_Sales"].append(m.Demand[i].value)
    data["Opt_Revenue"].append(m.Revenue[i].value)
    data["Final_Inv"].append(m.FinalInv[i].value)

pd.DataFrame(data).round(3)

,ITEM,Opt_Discount,Opt_Price,Opt_Sales,Opt_Revenue,Final_Inv
0,A,0.531,11.250,6.375,71.719,5.625
1,B,0.523,15.250,12.563,191.578,11.437
2,C,0.667,13.333,24.000,320.000,12.000


### Explanation of Results

The optimal discounts are 53.13% for A, 52.34% for B, and 66.67% for C, resulting in a net revenue of approximately $539.70.

All constraints are satisfied, with item C reaching the 12 lb inventory limit. IPOPT reports a locally optimal solution, which is also globally optimal for this model.

# Problem 4 — Four-week clearance plan

#### Load packages

In [175]:
import pandas as pd
from pyomo.environ import *
from pyomo.opt import SolverFactory

### Create data

In [176]:
items_df = pd.DataFrame(
    {"ITEM": ["A", "B", "C"], "basePrice": [24, 32, 40], "initInv": [12, 24, 36]}
)

weeks_df = pd.DataFrame({"WEEK": [1, 2, 3, 4]})

scalars_df = pd.DataFrame(
    {"Param": ["maxPct", "cap", "dispCost"], "Value": [0.80, 12, 1.50]}
)

print(items_df, weeks_df, scalars_df, sep="\n\n")

  ITEM  basePrice  initInv
0    A         24       12
1    B         32       24
2    C         40       36

   WEEK
0     1
1     2
2     3
3     4

      Param  Value
0    maxPct    0.8
1       cap   12.0
2  dispCost    1.5


### Define model

In [177]:
# define model
m = AbstractModel()

# define sets
m.ITEMS = Set()
m.WEEKS = Set(ordered=True)

# define parameters
m.basePrice = Param(m.ITEMS)
m.initInv = Param(m.ITEMS)
m.maxPct = Param()
m.cap = Param()
m.dispCost = Param()

# define decision variable
m.MarkPct = Var(
    m.ITEMS, m.WEEKS, domain=NonNegativeReals, bounds=(0, 1), initialize=0.25
)

# define implicit variables
m.MarkPrice = Var(m.ITEMS, m.WEEKS, domain=NonNegativeReals)
m.Demand = Var(m.ITEMS, m.WEEKS, domain=NonNegativeReals)
m.Revenue = Var(m.ITEMS, m.WEEKS, domain=NonNegativeReals)
m.FinalInv = Var(m.ITEMS, m.WEEKS, domain=NonNegativeReals)


# define markdown price
def Define_MarkPrice_Rule(m, i, w):
    return (1 - m.MarkPct[i, w]) * m.basePrice[i] == m.MarkPrice[i, w]

m.Define_MarkPrice = Constraint(m.ITEMS, m.WEEKS, rule=Define_MarkPrice_Rule)


# define demand using original initial inventory
def Define_Demand_Rule(m, i, w):
    return m.MarkPct[i, w] * m.initInv[i] == m.Demand[i, w]

m.Define_Demand = Constraint(m.ITEMS, m.WEEKS, rule=Define_Demand_Rule)


# define revenue
def Define_Revenue_Rule(m, i, w):
    return m.MarkPrice[i, w] * m.Demand[i, w] == m.Revenue[i, w]

m.Define_Revenue = Constraint(m.ITEMS, m.WEEKS, rule=Define_Revenue_Rule)


# define inventory carried between weeks
def Define_FinalInv_Rule(m, i, w):
    if w == m.WEEKS.first():
        return m.FinalInv[i, w] == m.initInv[i] - m.Demand[i, w]
    else:
        return m.FinalInv[i, w] == m.FinalInv[i, m.WEEKS.prev(w)] - m.Demand[i, w]

m.Define_FinalInv = Constraint(m.ITEMS, m.WEEKS, rule=Define_FinalInv_Rule)


# define maximum discount
def Max_Discount_Rule(m, i, w):
    return m.MarkPct[i, w] <= m.maxPct

m.Max_Discount = Constraint(m.ITEMS, m.WEEKS, rule=Max_Discount_Rule)


# limit inventory only at the end of the final week
def Leftover_Inv_Rule(m, i):
    return m.FinalInv[i, m.WEEKS.last()] <= m.cap

m.Leftover_Inv = Constraint(m.ITEMS, rule=Leftover_Inv_Rule)


# define objective function
def Max_Revenue_Rule(m):
    return sum(m.Revenue[i, w] for i in m.ITEMS for w in m.WEEKS) - sum(
        m.dispCost * m.FinalInv[i, m.WEEKS.last()] for i in m.ITEMS
    )

m.Max_Revenue = Objective(rule=Max_Revenue_Rule, sense=maximize)

### Create instance

In [178]:
scalars = scalars_df.set_index("Param")["Value"].to_dict()

instanceData = {
    None: {
        "ITEMS": {None: items_df["ITEM"].tolist()},
        "WEEKS": {None: weeks_df["WEEK"].tolist()},
        "basePrice": items_df.set_index("ITEM")["basePrice"].to_dict(),
        "initInv": items_df.set_index("ITEM")["initInv"].to_dict(),
        "maxPct": {None: scalars["maxPct"]},
        "cap": {None: scalars["cap"]},
        "dispCost": {None: scalars["dispCost"]},
    }
}

instance = m.create_instance(instanceData)

### Solve

In [179]:
solver = SolverFactory("ipopt")
solution = solver.solve(instance)

assert check_optimal_termination(solution), "Solver did not terminate optimally."

### Print solution

In [180]:
print(solution["Solver"])

data = {
    "ITEM": [],
    "WEEK": [],
    "Opt_Discount": [],
    "Opt_Price": [],
    "Opt_Demand": [],
    "Opt_Revenue": [],
    "Final_Inv": [],
}

for i in instance.ITEMS:
    for w in instance.WEEKS:
        data["ITEM"].append(i)
        data["WEEK"].append(w)
        data["Opt_Discount"].append(value(instance.MarkPct[i, w]))
        data["Opt_Price"].append(value(instance.MarkPrice[i, w]))
        data["Opt_Demand"].append(value(instance.Demand[i, w]))
        data["Opt_Revenue"].append(value(instance.Revenue[i, w]))
        data["Final_Inv"].append(value(instance.FinalInv[i, w]))

output = pd.DataFrame(data)

print(output.round(4).to_string(index=False))
print(f"Total Net Revenue: ${value(instance.Max_Revenue):,.2f}")


- Status: ok
  Message: Ipopt 3.14.20\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.70455002784729

ITEM  WEEK  Opt_Discount  Opt_Price  Opt_Demand  Opt_Revenue  Final_Inv
   A     1          0.25       18.0         3.0         54.0        9.0
   A     2          0.25       18.0         3.0         54.0        6.0
   A     3          0.25       18.0         3.0         54.0        3.0
   A     4          0.25       18.0         3.0         54.0       -0.0
   B     1          0.25       24.0         6.0        144.0       18.0
   B     2          0.25       24.0         6.0        144.0       12.0
   B     3          0.25       24.0         6.0        144.0        6.0
   B     4          0.25       24.0         6.0        144.0       -0.0
   C     1          0.25       30.0         9.0        270.0       27.0
   C     2          0.25       30.0         9.0        270.0       18.0
   C     3          0.25       30.0         9.0        270.0 

#### Explanation of results

The shop should discount every item by 25% each week. Weekly prices are
$18, $24, and $30 for A, B, and C, with weekly sales of 3, 6, and 9 lb.
All inventory is sold by week 4, giving net revenue of $1,872 with no
disposal expense.

Final-week stock availability is binding for each item. The 12-lb
leftover limits and 80% maximum discounts are not binding.

IPOPT reports optimal termination. Although IPOPT does not generally
certify global optimality, this model reduces to a concave quadratic
objective with linear constraints, so its optimum is global.

# Problem 5 — Limited markdown selections

### Create data

In [181]:
instanceData = {
    None: {
        "ITEMS": {None: ["A", "B", "C"]},
        "basePrice": dict(zip(["A", "B", "C"], [24, 32, 40])),
        "initInv": dict(zip(["A", "B", "C"], [12, 24, 36])),
        "maxPct": {None: 0.80},
        "cap": {None: 12},
        "dispCost": {None: 1.50},
        "maxSelected": {None: 2},
    }
}

### Define model

In [182]:
m = AbstractModel()

# define sets
m.ITEMS = Set()

# define parameters
m.basePrice = Param(m.ITEMS)
m.initInv = Param(m.ITEMS)
m.maxPct = Param()
m.cap = Param()
m.dispCost = Param()
m.maxSelected = Param()

# define decision variables
m.Selected = Var(m.ITEMS, domain=Binary)
m.MarkPct = Var(m.ITEMS, domain=NonNegativeReals, initialize=0.60)

# define implicit variables
m.MarkPrice = Var(m.ITEMS, domain=NonNegativeReals)
m.Demand = Var(m.ITEMS, domain=NonNegativeReals)
m.Revenue = Var(m.ITEMS, domain=NonNegativeReals)
m.FinalInv = Var(m.ITEMS, domain=NonNegativeReals)

# define markdown price
def MarkPrice_Rule(m, i):
    return m.MarkPrice[i] == m.basePrice[i] * (1 - m.MarkPct[i])

m.Define_MarkPrice = Constraint(m.ITEMS, rule=MarkPrice_Rule)

# define demand
def Demand_Rule(m, i):
    return m.Demand[i] == m.initInv[i] * m.MarkPct[i]

m.Define_Demand = Constraint(m.ITEMS, rule=Demand_Rule)

# define revenue
def Revenue_Rule(m, i):
    return m.Revenue[i] == m.MarkPrice[i] * m.Demand[i]

m.Define_Revenue = Constraint(m.ITEMS, rule=Revenue_Rule)

# define ending inventory
def FinalInv_Rule(m, i):
    return m.FinalInv[i] == m.initInv[i] - m.Demand[i]

m.Define_FinalInv = Constraint(m.ITEMS, rule=FinalInv_Rule)

# link discount to selection
def Max_Discount_Rule(m, i):
    return m.MarkPct[i] <= m.maxPct * m.Selected[i]

m.Max_Discount = Constraint(m.ITEMS, rule=Max_Discount_Rule)

# limit the number of selected items
def Selection_Limit_Rule(m):
    return sum(m.Selected[i] for i in m.ITEMS) <= m.maxSelected

m.Selection_Limit = Constraint(rule=Selection_Limit_Rule)

# limit leftover inventory for each item
def Inventory_Cap_Rule(m, i):
    return m.FinalInv[i] <= m.cap

m.Inventory_Cap = Constraint(m.ITEMS, rule=Inventory_Cap_Rule)

# define objective function
def Max_Revenue_Rule(m):
    return sum(m.Revenue[i] - m.dispCost * m.FinalInv[i] for i in m.ITEMS)

m.Max_Revenue = Objective(rule=Max_Revenue_Rule, sense=maximize)

### Create instance

In [183]:
instance = m.create_instance(instanceData)

### Solve

B and C must be selected to meet their leftover limits. The two-item
selection limit therefore leaves A unselected. Fix these required
selections before solving the continuous pricing problem with IPOPT.

In [184]:
# fix selections required by the inventory and selection limits
instance.Selected["A"].fix(0)
instance.Selected["B"].fix(1)
instance.Selected["C"].fix(1)

solver = SolverFactory("ipopt")
solution = solver.solve(instance)

assert check_optimal_termination(solution), "Solver did not terminate optimally."

### Print solution

In [185]:
print(solution['Solver'])

output = pd.DataFrame([
    {
        'Item': i,
        **{
            name: value(getattr(instance, name)[i])
            for name in [
                'Selected', 'MarkPct', 'MarkPrice',
                'Demand', 'Revenue', 'FinalInv'
            ]
        },
        'NetRevenue': value(
            instance.Revenue[i]
            - instance.dispCost * instance.FinalInv[i]
        )
    }
    for i in instance.ITEMS
])

print(output.round(4).to_string(index=False))
print(f'Total Net Revenue: ${value(instance.Max_Revenue):,.2f}')


- Status: ok
  Message: Ipopt 3.14.20\x3a Optimal Solution Found
  Termination condition: optimal
  Id: 0
  Error rc: 0
  Time: 0.4434490203857422

Item  Selected  MarkPct  MarkPrice  Demand  Revenue  FinalInv  NetRevenue
   A         0   0.0000    24.0000  0.0000   0.0000   12.0000    -18.0000
   B         1   0.5234    15.2500 12.5625 191.5781   11.4375    174.4219
   C         1   0.6667    13.3333 24.0000 320.0000   12.0000    302.0000
Total Net Revenue: $458.42


##### Explanation of results

The shop should select B and C for markdown. Discount B by about
52.3438% and C by 66.6667%, leaving A at its original price. Sales are
0, 12.5625, and 24 lb, leaving 12, 11.4375, and 12 lb for A, B, and C.
Total net revenue is $458.42.

The two-item selection limit and the leftover limits for A and C are
binding. No discount reaches the 80% maximum.

IPOPT reports optimal termination for the continuous problem. The
selection decisions are forced by feasibility, and the remaining
problem reduces to a concave quadratic objective with linear
constraints, establishing global optimality.